In [0]:
--  set catalog + schema
USE CATALOG ymotorna_DB;
USE SCHEMA superstore_lakehouse;

-- check
SELECT current_catalog(), 
    current_schema(); 

current_catalog(),current_schema()
ymotorna_db,superstore_lakehouse


In [0]:
-- Top 5 products by revenue
select product_id,
    product_name,
    category,
    sub_category,
    sum(total_revenue) as total_revenue,
    sum(quantity_sold) as quantity_sold
from gold_products
group by 1,2,3,4
order by total_revenue desc
limit 5;

product_id,product_name,category,sub_category,total_revenue,quantity_sold
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,technology,copiers,61599.82400000001,20
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,office supplies,binders,27453.384,31
TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,technology,machines,22638.48,6
FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,furniture,chairs,21870.576,39
OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,office supplies,binders,19823.479,37


In [0]:
-- Top region by revenue
select * from gold_sales_region limit 5;

region,revenue
west,713471.3445000004
east,672515.9939999981
central,497800.8728000007
south,388983.58500000037


In [0]:
-- Monthly revenue trend
with cte as (
    select date_format(date, 'yyyy-MM') as ymo,
        sum(revenue) as total_revenue,
        lag(sum(revenue), 1) over (order by date_format(date, 'yyyy-MM')) as prev_month_rev
    from gold_sales_daily
    group by date_format(date, 'yyyy-MM')
)

select *,
    total_revenue - prev_month_rev as monthly_growth
from cte
order by ymo asc;

ymo,total_revenue,prev_month_rev,monthly_growth
2014-01,14161.349,null,null
2014-02,4119.816000000001,14161.349,-10041.533
2014-03,55526.19899999998,4119.816000000001,51406.38299999998
2014-04,28139.560999999998,55526.19899999998,-27386.63799999998
2014-05,23634.666999999998,28139.560999999998,-4504.894
2014-06,34508.9956,23634.666999999998,10874.328600000004
2014-07,33500.873,34508.9956,-1008.1226000000024
2014-08,27603.512500000004,33500.873,-5897.360499999995
2014-09,81496.8068,27603.512500000004,53893.2943
2014-10,31394.941,81496.8068,-50101.86580000001


In [0]:
-- Running revenue total
select date,
    sum(revenue) over (order by date
                    rows between unbounded preceding and current row) as running_total
from gold_sales_daily
order by date asc;

date,running_total
2014-01-04,288.06
2014-01-05,307.596
2014-01-06,4714.696
2014-01-07,4801.854
2014-01-09,4842.398
2014-01-10,4897.228
2014-01-11,4907.168
2014-01-13,8460.963
2014-01-14,8522.922999999999
2014-01-15,8672.873


In [0]:
-- Top product per region
with cte as (
    select p.region,
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        sum(p.total_revenue) as total_revenue,
        row_number() over(partition by p.region order by sum(p.total_revenue) desc) as product_rev_rank,
        r.revenue as total_region_rev
    from gold_products p
    join gold_sales_region r 
    on p.region = r.region
    group by 1,2,3,4,5, 8
)

select *,
    round(total_revenue/total_region_rev*100, 2) as pct_of_region_rev
from cte
where product_rev_rank = 1;


region,product_id,product_name,category,sub_category,total_revenue,product_rev_rank,total_region_rev,pct_of_region_rev
central,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,technology,copiers,17499.95,1,497800.8728000007,3.52
east,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,technology,copiers,30099.914000000004,1,672515.9939999981,4.48
south,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,technology,machines,22638.48,1,388983.58500000037,5.82
west,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,technology,copiers,13999.96,1,713471.3445000004,1.96


---
### Performance Analysis

#### Baseline query

In [0]:
%python

baseline_df = spark.sql("""
    with cte as (
    select p.region,
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        sum(p.total_revenue) as total_revenue,
        row_number() over(partition by p.region order by sum(p.total_revenue) desc) as product_rev_rank,
        r.revenue as total_region_rev
    from gold_products p
    join gold_sales_region r 
    on p.region = r.region
    group by 1,2,3,4,5, 8
    )

    select *,
        round(total_revenue/total_region_rev*100, 2) as pct_of_region_rev
    from cte
    where product_rev_rank = 1;
""")

baseline_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (28)
+- == Initial Plan ==
   PhotonResultStage (27)
   +- PhotonColumnarToRow (26)
      +- PhotonProject (25)
         +- PhotonFilter (24)
            +- PhotonWindow (23)
               +- PhotonTopK (22)
                  +- PhotonShuffleExchangeSource (21)
                     +- PhotonShuffleMapStage (20)
                        +- PhotonShuffleExchangeSink (19)
                           +- PhotonProject (18)
                              +- PhotonFilter (17)
                                 +- PhotonWindow (16)
                                    +- PhotonTopK (15)
                                       +- PhotonProject (14)
                                          +- PhotonGroupingAgg (13)
                                             +- PhotonShuffleExchangeSource (12)
                                                +- PhotonShuffleMapStage (11)
                                                   +- PhotonShuffleExchangeSink (10)
        

#### Optimised query (with broadcast())

In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast, row_number
from pyspark.sql.window import Window

window_spec = Window.partitionBy("region").orderBy(F.col("total_revenue").desc())

optimized_df = (
    spark.table("gold_products")
    .groupBy("region", "product_id", "product_name", "category", "sub_category")
    .agg(F.sum("total_revenue").alias("total_revenue"))
    .join(
        broadcast(spark.table("gold_sales_region").select("region", "revenue")),
        on="region"
    )
    .withColumnRenamed("revenue", "total_region_rev")
    .withColumn("product_rev_rank", row_number().over(window_spec))
    .filter(F.col("product_rev_rank")==1)
    .withColumn("pct_of_region_rev", F.round(F.col("total_revenue") / F.col("total_region_rev") *100, 2))
)

optimized_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (26)
+- == Initial Plan ==
   PhotonResultStage (25)
   +- PhotonColumnarToRow (24)
      +- PhotonProject (23)
         +- PhotonFilter (22)
            +- PhotonWindow (21)
               +- PhotonTopK (20)
                  +- PhotonShuffleExchangeSource (19)
                     +- PhotonShuffleMapStage (18)
                        +- PhotonShuffleExchangeSink (17)
                           +- PhotonProject (16)
                              +- PhotonFilter (15)
                                 +- PhotonWindow (14)
                                    +- PhotonTopK (13)
                                       +- PhotonProject (12)
                                          +- PhotonBroadcastHashJoin Inner (11)
                                             :- PhotonGroupingAgg (6)
                                             :  +- PhotonShuffleExchangeSource (5)
                                             :     +- PhotonShuffleMapStage (4)
       

#### Time comparison

In [0]:
%python

import time

start = time.time()
baseline_df.count()
base_time = round(time.time() - start, 2)
print(f"Baseline: {base_time}s")

start = time.time()
optimized_df.count()
opt_time = round(time.time() - start, 2)
print(f"Optimized: {opt_time}s")

Baseline: 1.24s
Optimized: 0.85s
